# P30 · Detector operating point (chosen on validation) + test P/R/F1 and false-positive breakdown

Closes two Phase 2 gaps for M4: *"operating point justified by a threshold sweep"* and *"error breakdown"*.

- The frozen detector (`Joint_StrongerAug/best.pt`, 640 px, NMS 0.70) is run on **validation only**.
- The threshold is chosen on validation by a rule fixed in advance: **max F2** (recall-oriented), grid 0.05–0.95.
- Test numbers come from the **saved P09 test predictions**; the model is not run on test. A check first confirms that the matching reproduces the P09 recall at conf 0.001 (0.548 sea, 0.808 urban).

**Runtime → T4 GPU, then Run all** (about 10–20 minutes). Send the last printed tables.

In [ ]:
!pip -q install ultralytics==8.4.215
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# P30 · Detector operating point chosen on VALIDATION, then reported on test from the SAVED P09 predictions.
# Rule fixed before running: per domain, the confidence threshold on the grid 0.05..0.95 (step 0.05) that
# maximises F2 (recall-oriented, beta = 2) on validation; ties -> the lower threshold. NMS IoU 0.70, 640 px, frozen best.pt.
# Matching (same definition as the P09 recall-by-size analysis): one-to-one, class-consistent, IoU >= 0.5,
# predictions taken in descending confidence. The model is NOT run on test: test numbers come from the P09 files.
import os, glob, gzip, json, csv, datetime, numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

cands = [p for pat in ['/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial',
                       '/content/drive/MyDrive/*/AI251P_Capstone_CrossDomainAerial',
                       '/content/drive/.shortcut-targets-by-id/*/AI251P_Capstone_CrossDomainAerial'] for p in glob.glob(pat)
         if os.path.isdir(os.path.join(p, '03_Detection'))]
assert cands, 'project folder not found'
R = Path(cands[0]) / '03_Detection'
DATA = R / '2026-11-02_YOLO_Prepared_v1'
P09 = R / '2026-11-06_P09_Evaluation'
MODEL = R / '2026-11-04_P08_Baseline/YOLO11n_Joint_StrongerAug/weights/best.pt'
OUT = P09 / 'P30_OperatingPoint'; OUT.mkdir(exist_ok=True)
assert MODEL.is_file(), MODEL
assert not (OUT / 'P30_TestAtOperatingPoint_v1.csv').exists(), 'P30 already done: do not repeat'
GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)
NAMES = ['swimmer', 'boat', 'jetski', 'life_saving_appliance', 'buoy', 'car', 'truck', 'bus']
EXPECT = {'sea': {'val': 1153, 'test': 1869}, 'urban': {'val': 699, 'test': 1038}}
RECALL_P09 = {'sea': 0.548, 'urban': 0.808}     # object-level recall at conf 0.001 reported in P09 (recall by size)
print('ROOT', R)

def image_list(domain, split):
    names = [Path(l.strip()).name for l in (DATA / 'domain_lists' / f'{domain}_{split}.txt').read_text().splitlines() if l.strip()]
    assert len(names) == EXPECT[domain][split], (domain, split, len(names))
    return names

def gt_boxes(split, name, w, h):
    f = DATA / 'labels' / split / (Path(name).stem + '.txt')
    if not f.exists(): return np.zeros((0,)), np.zeros((0, 4))
    a = np.loadtxt(f, ndmin=2)
    if a.size == 0: return np.zeros((0,)), np.zeros((0, 4))
    c, xc, yc, bw, bh = a[:, 0].astype(int), a[:, 1] * w, a[:, 2] * h, a[:, 3] * w, a[:, 4] * h
    return c, np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], 1)

def iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(gc, gb, pc, pb, ps, thr):
    """returns tp flags per kept prediction, matched-gt flags, FP type per prediction"""
    keep = ps >= thr; pc, pb, ps = pc[keep], pb[keep], ps[keep]
    order = np.argsort(-ps); pc, pb = pc[order], pb[order]
    M = iou(pb, gb); used = np.zeros(len(gc), bool); tp = np.zeros(len(pc), bool); kind = np.array([''] * len(pc), dtype=object)
    for i in range(len(pc)):
        same = (gc == pc[i]) & ~used & (M[i] >= 0.5) if len(gc) else np.zeros(0, bool)
        if same.any():
            j = np.flatnonzero(same)[np.argmax(M[i][same])]; used[j] = True; tp[i] = True; continue
        m = M[i] if len(gc) else np.zeros(0)
        if len(gc) and ((gc == pc[i]) & (m >= 0.5)).any(): kind[i] = 'duplicate'
        elif len(gc) and ((gc != pc[i]) & (m >= 0.5)).any(): kind[i] = 'wrong class'
        elif len(gc) and ((gc == pc[i]) & (m >= 0.1)).any(): kind[i] = 'localization'
        else: kind[i] = 'background'
    return tp, used, kind

def evaluate(records, split, thr):
    TP = FP = FN = 0; kinds = {}; per = {n: [0, 0, 0] for n in NAMES}
    for r in records:
        gc, gb = gt_boxes(split, r['image'], r['width'], r['height'])
        P = r['predictions']
        pc = np.array([p['cls'] for p in P], int); ps = np.array([p['conf'] for p in P], float)
        pb = np.array([p['box'] for p in P], float).reshape(-1, 4)
        tp, used, kind = match(gc, gb, pc, pb, ps, thr)
        TP += tp.sum(); FP += (~tp).sum(); FN += (~used).sum()
        for k in kind[~tp]: kinds[k] = kinds.get(k, 0) + 1
        keep = ps >= thr; pcs = pc[keep][np.argsort(-ps[keep])]
        for c in range(8):
            per[NAMES[c]][0] += int((tp & (pcs == c)).sum()); per[NAMES[c]][1] += int((~tp & (pcs == c)).sum())
            per[NAMES[c]][2] += int((~used & (gc == c)).sum())
    p = TP / max(TP + FP, 1); r_ = TP / max(TP + FN, 1)
    f1 = 2 * p * r_ / max(p + r_, 1e-12); f2 = 5 * p * r_ / max(4 * p + r_, 1e-12)
    return dict(thr=thr, tp=int(TP), fp=int(FP), fn=int(FN), precision=p, recall=r_, f1=f1, f2=f2), kinds, per

In [ ]:
model = YOLO(str(MODEL))
VAL = {}
for d in ['sea', 'urban']:
    cache = OUT / f'P30_ValPredictions_{d}_v1.json.gz'
    if cache.exists():
        VAL[d] = json.load(gzip.open(cache, 'rt')); print(d, 'val predictions loaded', len(VAL[d])); continue
    recs = []
    for k, name in enumerate(image_list(d, 'val')):
        img = DATA / 'images' / 'val' / name; assert img.exists(), img
        res = model.predict(str(img), imgsz=640, conf=0.001, iou=0.7, max_det=300, verbose=False)[0]
        b = res.boxes
        recs.append({'image': name, 'height': int(res.orig_shape[0]), 'width': int(res.orig_shape[1]),
                     'predictions': [{'cls': int(c), 'conf': float(s), 'box': x.tolist()}
                                     for x, s, c in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy())]})
        if k % 200 == 0: print(d, k, flush=True)
    json.dump(recs, gzip.open(cache, 'wt')); VAL[d] = recs; print(d, 'val predictions saved', len(recs))

In [ ]:
from functools import lru_cache
_gt_raw = gt_boxes
@lru_cache(maxsize=None)
def gt_boxes(split, name, w, h):
    return _gt_raw(split, name, w, h)
print('labels cached')
SWEEP, CHOSEN = [], {}
for d in ['sea', 'urban']:
    rows = [evaluate(VAL[d], 'val', t)[0] for t in GRID]
    for r in rows: r['domain'] = d
    SWEEP += rows
    best = max(rows, key=lambda r: (round(r['f2'], 6), -r['thr']))
    CHOSEN[d] = best['thr']; print(f"{d}: chosen on val tau_conf = {best['thr']}  (P {best['precision']:.3f}, R {best['recall']:.3f}, F1 {best['f1']:.3f}, F2 {best['f2']:.3f})")
pd.DataFrame(SWEEP).to_csv(OUT / 'P30_ValSweep_v1.csv', index=False)
json.dump({'rule': 'max F2 on val, grid 0.05..0.95 step 0.05, ties -> lower', 'nms_iou': 0.7, 'imgsz': 640,
           'chosen': CHOSEN, 'created_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M')},
          open(OUT / 'P30_OperatingPoint_v1.json', 'w'), indent=2)

In [ ]:
when = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M')
with open(OUT / 'P30_TestUseLog.csv', 'a', newline='') as fh:
    csv.writer(fh).writerow([when, 'P30', 'saved P09 test predictions (Joint_StrongerAug, conf 0.001)',
                             f'report-only: P/R/F1 and FP breakdown at the val-chosen thresholds {CHOSEN}; nothing tuned'])
ROWS, FPK, PER = [], [], []
for d in ['sea', 'urban']:
    files = sorted((P09 / 'predictions' / 'Joint_StrongerAug' / d).glob('*.json.gz'))
    recs = [r for f in files for r in json.load(gzip.open(f, 'rt'))]
    names = {r['image'] for r in recs}; assert len(names) == EXPECT[d]['test'] == len(recs), (d, len(recs), len(names))
    base = evaluate(recs, 'test', 0.001)[0]
    assert abs(base['recall'] - RECALL_P09[d]) < 0.003, f"{d}: recall at 0.001 {base['recall']:.4f} != P09 {RECALL_P09[d]}: matching differs, stop"
    m, kinds, per = evaluate(recs, 'test', CHOSEN[d])
    ROWS += [{'domain': d, 'setting': 'conf 0.001 (P09 check)', **base}, {'domain': d, 'setting': 'operating point (val)', **m}]
    FPK += [{'domain': d, 'type': k, 'count': v, 'share': round(v / max(m['fp'], 1), 3)} for k, v in sorted(kinds.items())]
    PER += [{'domain': d, 'class': c, 'tp': v[0], 'fp': v[1], 'fn': v[2],
             'precision': v[0] / max(v[0] + v[1], 1), 'recall': v[0] / max(v[0] + v[2], 1)} for c, v in per.items() if v[0] + v[2] > 0]
pd.DataFrame(ROWS).round(4).to_csv(OUT / 'P30_TestAtOperatingPoint_v1.csv', index=False)
pd.DataFrame(FPK).to_csv(OUT / 'P30_TestFPBreakdown_v1.csv', index=False)
pd.DataFrame(PER).round(4).to_csv(OUT / 'P30_TestPerClass_v1.csv', index=False)
pd.set_option('display.width', 200)
print(pd.DataFrame(ROWS).round(4).to_string(index=False)); print(); print(pd.DataFrame(FPK).to_string(index=False))
print(); print(pd.DataFrame(PER).round(3).to_string(index=False))
print('\nsaved in', OUT)